# ЛР2. Базовая модель SMS-спам-фильтра и проверка входов

Ноутбук вызывает функции из `src/`: загрузка и проверка данных (`src.data`), обучение (`src.train`), предсказание с валидацией (`src.predict`). Модель в ноутбуке обучается заново только для демонстрации; сохранённый артефакт создаётся командой `python -m src.train`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, skops
from src import config
from src.data import verify_raw_data, load_raw, check_raw, prepare, data_version, normalize_key
from src.train import build_pipeline, split, evaluate
from src.predict import preprocess, predict, load_model, InputValidationError
print("python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| skops", skops.__version__, "| pandas", pd.__version__)

python 3.14.4 | scikit-learn 1.9.1 | skops 0.16.0 | pandas 3.1.0


## 1. Целостность и качество данных

In [2]:
raw_sha = verify_raw_data()          # исключение, если SHA-256 не совпадает с эталоном ЛР1
print("SHA-256 совпадает с эталоном ЛР1:", raw_sha)
raw = load_raw()
checks = check_raw(raw)
print(json.dumps(checks, ensure_ascii=False, indent=1))
print(raw.dtypes)

SHA-256 совпадает с эталоном ЛР1: 7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d
{
 "rows": 5574,
 "columns": [
  "label",
  "text"
 ],
 "missing": {
  "label": 0,
  "text": 0
 },
 "labels": [
  "ham",
  "spam"
 ],
 "labels_valid": true,
 "blank_texts": 0,
 "exact_duplicates": 403,
 "normalized_duplicates": 415,
 "conflicting_labels": 0,
 "class_counts": {
  "ham": 4827,
  "spam": 747
 }
}
label    str
text     str
dtype: object


In [3]:
key = normalize_key(raw["text"])
dups = raw[key.duplicated(keep=False)].assign(key=key).sort_values("key")
print("Сообщений, входящих в группы дубликатов:", len(dups))
print("Удаляется (повторы после первого):", key.duplicated().sum())
print("Из них spam:", raw[key.duplicated()]["label"].eq("spam").sum())
dups.groupby("key").size().sort_values(ascending=False).head(5)

Сообщений, входящих в группы дубликатов: 705
Удаляется (повторы после первого): 415
Из них spam: 105


key
sorry, i'll call later                                                                                                                                                 30
i cant pick the phone right now. pls send a message                                                                                                                    12
ok...                                                                                                                                                                  10
say this slowly.? god,i love you &amp; i need you,clean my heart with your blood.send this to ten special people &amp; u c miracle tomorrow, do it,pls,pls do it...     4
ok                                                                                                                                                                      4
dtype: int64

In [4]:
df = prepare(raw)
version = data_version(raw_sha, df)
print(json.dumps(version, ensure_ascii=False, indent=1))
counts = df["label"].value_counts()
print(pd.DataFrame({"count": counts, "share_%": (counts / len(df) * 100).round(2)}))
print(f"ham : spam = {counts['ham'] / counts['spam']:.2f} : 1")

{
 "data_id": "smsspam-raw-7d039a24a608-dedup-97a39c090cb8",
 "raw_sha256": "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d",
 "prepared_sha256": "97a39c090cb88408adc8166cc16648a138d8a9f6e9121016d5d0ee02e7ccb8ca",
 "dedup_rule": "lower+collapse-whitespace",
 "rows_prepared": 5159
}
       count  share_%
label                
ham     4517    87.56
spam     642    12.44
ham : spam = 7.04 : 1


## 2. Достаточность данных

In [5]:
df["length"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()
print(df.groupby("label")[["length", "n_words"]].describe().round(1).T)
print("Сообщений длиннее 1000 символов:", (df["length"] > 1000).sum(), "| max:", df["length"].max())
non_ascii = df["text"].map(lambda s: any(ord(c) > 127 for c in s))
print("Сообщений с не-ASCII символами:", non_ascii.sum())
print("Сообщений с URL/доменом:", df["text"].str.contains(r"https?://|www\.|\.com", case=False).sum())

label             ham   spam
length  count  4517.0  642.0
        mean     70.9  137.5
        std      56.6   30.0
        min       2.0   13.0
        25%      34.0  131.0
        50%      53.0  148.0
        75%      91.0  157.0
        max     910.0  223.0
n_words count  4517.0  642.0
        mean     14.2   23.7
        std      11.2    6.0
        min       1.0    2.0
        25%       7.0   21.2
        50%      11.0   25.0
        75%      19.0   28.0
        max     171.0   35.0
Сообщений длиннее 1000 символов: 0 | max: 910
Сообщений с не-ASCII символами: 440
Сообщений с URL/доменом: 115


## 3. Разделение выборок

In [6]:
train_df, test_df = split(df)
print(f"train: {len(train_df)} (spam {train_df['y'].mean():.2%}) | test: {len(test_df)} (spam {test_df['y'].mean():.2%})")
assert not set(normalize_key(train_df["text"])) & set(normalize_key(test_df["text"])), "утечка дубликатов"
print("Пересечений текстов train/test нет")
X_train, y_train = preprocess(train_df["text"].tolist()), train_df["y"].to_numpy()
X_test, y_test = preprocess(test_df["text"].tolist()), test_df["y"].to_numpy()

train: 4127 (spam 12.45%) | test: 1032 (spam 12.40%)
Пересечений текстов train/test нет


## 4. Обучение: подбор C кросс-валидацией только на train

In [7]:
import time
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier
cv = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.SEED)
search = GridSearchCV(build_pipeline(), {"clf__C": list(config.C_GRID)}, scoring="f1", cv=cv)
t0 = time.perf_counter(); search.fit(X_train, y_train); t_search = time.perf_counter() - t0
print(pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]].round(4))
best_c = search.best_params_["clf__C"]
model = build_pipeline().set_params(clf__C=best_c)
t0 = time.perf_counter(); model.fit(X_train, y_train); t_fit = time.perf_counter() - t0
print(f"C={best_c}; подбор {t_search:.2f} c; обучение {t_fit:.3f} c; словарь {len(model.named_steps['tfidf'].vocabulary_)} признаков")

   param_clf__C  mean_test_score  std_test_score
0           0.1           0.9206          0.0126
1           1.0           0.9330          0.0126
2          10.0           0.9416          0.0217
3         100.0           0.9336          0.0172
C=10.0; подбор 1.16 c; обучение 0.068 c; словарь 10074 признаков


## 5. Оценка на отложенной тестовой выборке (один раз)

In [8]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
y_pred = (proba >= config.DECISION_THRESHOLD).astype(int)
print("TF-IDF + LogisticRegression\n", classification_report(y_test, y_pred, target_names=config.LABELS, digits=4))
print("DummyClassifier (всегда ham)\n", classification_report(y_test, dummy.predict(X_test), target_names=config.LABELS, digits=4, zero_division=0))

TF-IDF + LogisticRegression
               precision    recall  f1-score   support

         ham     0.9923    0.9934    0.9928       904
        spam     0.9528    0.9453    0.9490       128

    accuracy                         0.9874      1032
   macro avg     0.9725    0.9693    0.9709      1032
weighted avg     0.9874    0.9874    0.9874      1032

DummyClassifier (всегда ham)
               precision    recall  f1-score   support

         ham     0.8760    1.0000    0.9339       904
        spam     0.0000    0.0000    0.0000       128

    accuracy                         0.8760      1032
   macro avg     0.4380    0.5000    0.4669      1032
weighted avg     0.7673    0.8760    0.8181      1032



In [15]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, yp, title in ((axes[0], dummy.predict(X_test), "DummyClassifier (всегда ham)"),
                      (axes[1], y_pred, f"TF-IDF + LogReg (C={best_c})")):
    ConfusionMatrixDisplay(confusion_matrix(y_test, yp, labels=[0, 1]), display_labels=config.LABELS).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title); ax.set_xlabel("Предсказанный класс"); ax.set_ylabel("Истинный класс")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "confusion_matrix.png", dpi=150); plt.show()

In [16]:
m = evaluate(y_test, y_pred)
per = pd.DataFrame(m["per_class"]).T[["precision", "recall", "f1"]]
ax = per.plot(kind="bar", rot=0, figsize=(6, 4), ylim=(0.8, 1.012), color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Метрики по классам на тестовой выборке"); ax.set_xlabel("Класс"); ax.set_ylabel("Значение")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.savefig(config.FIG_DIR / "per_class_metrics.png", dpi=150); plt.show()

### Ошибочные предсказания

In [17]:
pd.set_option("display.max_colwidth", 160)
err = test_df.assign(p_spam=proba.round(4), pred=y_pred)
err = err[err["pred"] != err["y"]][["row_id", "label", "p_spam", "text"]]
print("Ложноположительных (ham -> spam):", (err["label"] == "ham").sum(), "| ложноотрицательных (spam -> ham):", (err["label"] == "spam").sum())
err.sort_values("p_spam")

Ложноположительных (ham -> spam): 6 | ложноотрицательных (spam -> ham): 7


,row_id,label,p_spam,text
3173,3360,spam,0.0480,Sorry I missed your call let's talk when you have the time. I'm on 07090201529
5129,5542,spam,0.0929,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 DONT GIV A SHIT. BAILIFF DUE IN DAYS. I O £250 3 WANT £800
4979,5372,spam,0.1380,dating:i have had two of these. Only started after i sent a text to talk sport radio last week. Any connection do you think or coincidence?
4605,4949,spam,0.1415,"Hi this is Amy, we will be sending you a free phone number in a couple of days, which will give you an access to all the adult parties..."
2299,2402,spam,0.1463,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
2967,3132,spam,0.2462,"LookAtMe!: Thanks for your purchase of a video clip from LookAtMe!, you've been charged 35p. Think you can do better? Why not send a video in a MMSto 32323."
2768,2915,spam,0.4562,Sorry! U can not unsubscribe yet. THE MOB offer package has a min term of 54 weeks> pls resubmit request after expiry. Reply THEMOB HELP 4 more info
2196,2289,ham,0.5723,Dont you have message offer
3332,3536,ham,0.6341,I'm at home. Please call
3594,3824,ham,0.6347,"Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody."


## 6. Сохранённая модель: проверка хэша, загрузка, предсказание

In [18]:
saved = load_model()      # проверка SHA-256 -> проверка типов skops -> загрузка
print(open(config.MODEL_REFERENCE, encoding="utf-8").read())
baseline = json.loads((config.METRICS_DIR / "baseline.json").read_text(encoding="utf-8"))
print({k: baseline[k] for k in ("primary_metric", "value", "test_size", "data_id")})
print(predict("Congratulations! You have won a FREE prize. Call 09061701461 to claim", model=saved))
print(predict(["Are we still meeting for lunch tomorrow?", {"text": "URGENT! Claim your cash reward now"}], model=saved))

f8a60e7a786e4877b10a6e19b58ea1240d62290844d01b0850abc4fc6c1b2ef4  models/spam_tfidf_logreg.skops

{'primary_metric': 'f1_spam', 'value': 0.949, 'test_size': 1032, 'data_id': 'smsspam-raw-7d039a24a608-dedup-97a39c090cb8'}
{'label': 'spam', 'spam_probability': 0.9997}
[{'label': 'ham', 'spam_probability': 0.0159}, {'label': 'spam', 'spam_probability': 0.9895}]


In [19]:
for bad in ["", None, float("nan"), 42, "x" * 1001, b"\xff\xfe", {"msg": "hi"}]:
    try:
        predict(bad, model=saved)
    except InputValidationError as e:
        print(f"{repr(bad)[:30]:32} -> {e}")

''                               -> [empty] пустой текст
None                             -> [missing_value] значение отсутствует (None)
nan                              -> [wrong_type] ожидалась строка, получен float: nan
42                               -> [wrong_type] ожидалась строка, получен int: 42
'xxxxxxxxxxxxxxxxxxxxxxxxxxxxx   -> [too_long] длина 1001 символов превышает предел 1000
b'\xff\xfe'                      -> [bad_encoding] текст не является корректным UTF-8
{'msg': 'hi'}                    -> [missing_field] отсутствует обязательное поле 'text'


## 7. Эксперимент с устойчивостью (результаты `python -m src.robustness`)

In [22]:
rob = json.loads((config.METRICS_DIR / "robustness.json").read_text(encoding="utf-8"))
rows = {k: v for k, v in rob.items() if k in ("case", "spaces", "typos", "leet", "leet_all")}
print("Исходные 20 объектов:", rob["original"])
pd.DataFrame(rows).T[["flip_rate", "flips_spam_to_ham", "accuracy", "f1_spam", "mean_abs_delta_p", "mean_p_spam_class_after"]]

Исходные 20 объектов: {'accuracy': 0.95, 'f1_spam': 0.9474}


,flip_rate,flips_spam_to_ham,accuracy,f1_spam,mean_abs_delta_p,mean_p_spam_class_after
case,0.0,0.0,0.95,0.9474,0.0000,0.9208
spaces,0.0,0.0,0.95,0.9474,0.0000,0.9208
typos,0.0,0.0,0.95,0.9474,0.0185,0.8939
leet,0.0,0.0,0.95,0.9474,0.0271,0.8800
leet_all,0.1,2.0,0.85,0.8235,0.1633,0.6259


In [23]:
print("Доля незнакомых токенов на test:", rob["test_oov_share"])
print("P(spam) для пустого TF-IDF-вектора:", rob["empty_vector_spam_probability"])
pd.DataFrame(rob["atypical"]).T

Доля незнакомых токенов на test: {'mean': 0.1124, 'p95': 0.3714, 'max': 1.0, 'share_of_test_with_oov_1.0': 0.0048}
P(spam) для пустого TF-IDF-вектора: 0.0614


,text,label,spam_probability,oov_share
ru_text,"Поздравляем! Вы выиграли приз 10000 рублей, позвоните по номеру 88001234567",ham,0.0614,1.0
emoji_only,🎁💰📱🔥🔥,ham,0.0614,1.0
url_only,http://bit.ly/3xYz9Qw,ham,0.4563,0.5
digits_only,09061701461,ham,0.0614,1.0
